<a href="https://colab.research.google.com/github/lpastor75/digit_recognizer/blob/main/digit_recognizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Reconocimiento de Dígitos Manuscritos con Deep Learning

## Clasificación de imágenes MNIST mediante Redes Neuronales Convolucionales (CNN)

### Proyecto de Computer Vision y Deep Learning

**Autor:** Luis Pastor Nuevo  
**Tecnologías:** TensorFlow · Keras · NumPy · Pandas · Matplotlib · Scikit-learn  
**Dataset:** MNIST Handwritten Digits Dataset  
**Objetivo:** Construcción y optimización de modelos CNN para clasificación multiclase de imágenes.

---

## 📌 Descripción del proyecto

Este proyecto implementa un pipeline completo de Deep Learning aplicado al reconocimiento de dígitos manuscritos utilizando el dataset MNIST.

El notebook incluye:

- Exploración y análisis de datos
- Preprocesamiento y normalización
- Construcción de modelos CNN
- Experimentación y ajuste de hiperparámetros
- Técnicas de regularización
- Evaluación y comparación de modelos
- Generación de predicciones para Kaggle

El enfoque del proyecto está orientado a buenas prácticas de Machine Learning Engineering y experimentación reproducible.

In [ ]:
# ============================================
# LIBRERÍAS
# ============================================

import os
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# ============================================
# CONFIGURACIÓN GLOBAL
# ============================================

SEED = 101

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

In [ ]:
from google.colab import files

uploaded = files.upload()

## Instalación de dependencias

Este notebook instala automáticamente las librerías necesarias para garantizar reproducibilidad en cualquier entorno (Colab, local o Kaggle).

In [ ]:
# Instalación de dependencias
!pip install -r requirements.txt

In [ ]:
# ============================================
# CONFIGURACIÓN DEL PROYECTO
# ============================================

os.makedirs("images", exist_ok=True)

print("Carpeta images creada correctamente.")

# 📊 Dataset MNIST

El dataset MNIST es uno de los benchmarks clásicos en visión por computador y clasificación de imágenes.

Contiene imágenes en escala de grises de dígitos manuscritos del 0 al 9.

## Características principales

- 70.000 imágenes en total
- Resolución: 28x28 píxeles
- Clasificación multiclase (10 categorías)
- Imágenes en escala de grises

---

## Objetivo del modelo

Dado un dígito manuscrito, el modelo debe predecir correctamente la clase correspondiente.

Ejemplo:

| Imagen | Predicción |
|---|---|
| 7 manuscrito | Clase 7 |
| 3 manuscrito | Clase 3 |
| 0 manuscrito | Clase 0 |

# 📥 Descarga automática del dataset

El dataset se descarga automáticamente desde Kaggle.

Es necesario disponer de la API Key de Kaggle (`kaggle.json`).

In [ ]:
# ============================================
# DESCARGA DEL DATASET
# ============================================

!pip install -q kaggle

from google.colab import files

print("Sube tu archivo kaggle.json")
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c digit-recognizer

!mkdir -p data

!unzip -q digit-recognizer.zip -d data

print("Dataset descargado correctamente.")

In [ ]:
import zipfile

with zipfile.ZipFile("digit-recognizer.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

In [ ]:
# ============================================
# CARGA DEL DATASET
# ============================================

df_train = pd.read_csv("data/train.csv")
df_test = pd.read_csv("data/test.csv")

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

df_train.head()

# 🔎 Preparación de datos

El dataset de entrenamiento contiene:

- Una columna objetivo (`label`)
- 784 columnas con los valores de los píxeles

Cada imagen de 28x28 píxeles se almacena como un vector plano de 784 posiciones.

In [ ]:
# ============================================
# FEATURES Y TARGET
# ============================================

y_train = df_train["label"]
x_train = df_train.iloc[:, 1:]

# Conversión a arrays numpy
x_train = np.array(x_train)
y_train = np.array(y_train)

x_test = np.array(df_test)

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test:", x_test.shape)

## 🖼️ Visualización de imágenes

Antes de entrenar el modelo, se inspecciona una muestra aleatoria del dataset para verificar:

- Calidad de las imágenes
- Distribución visual
- Correcta carga de datos

In [ ]:
# ============================================
# VISUALIZACIÓN DE IMÁGENES
# ============================================

def show_images(images, save=False):

    fig = plt.figure(figsize=(8, 8))

    index = np.random.randint(len(images), size=100)

    for i in range(100):

        fig.add_subplot(10, 10, i + 1)

        plt.axis("off")

        plt.imshow(
            images[index[i]].reshape(28, 28),
            cmap="gray"
        )

    plt.tight_layout()

    if save:
        plt.savefig(
            "images/sample_digits.png",
            bbox_inches="tight"
        )

    plt.show()

In [ ]:
show_images(x_train, save=True)

# ⚙️ Preprocesamiento

Las imágenes contienen valores entre 0 y 255.

Para mejorar la estabilidad del entrenamiento y acelerar la convergencia, se normalizan los píxeles al rango [0, 1].

In [ ]:
# ============================================
# NORMALIZACIÓN
# ============================================

x_train = x_train / 255.0
x_test = x_test / 255.0

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

# Reshape para CNN
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

y_train = y_train.reshape(-1, 1)

print(x_train.shape)
print(x_test.shape)

In [ ]:
# ============================================
# TRAIN / VALIDATION SPLIT
# ============================================

X_train, X_val, Y_train, Y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.20,
    random_state=SEED
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)

# 🏗️ Modelo 1 · CNN Base

## Objetivo

Como punto de partida del proyecto, se implementa una Red Neuronal Convolucional (CNN) sencilla que servirá como modelo de referencia (*baseline*).

El objetivo de este modelo es establecer una línea base de rendimiento sobre la que comparar posteriores mejoras arquitectónicas y técnicas de regularización. La arquitectura combina capas convolucionales para la extracción automática de características espaciales y capas densas para la clasificación final de los dígitos.

Esta implementación permite evaluar la capacidad de una CNN básica para resolver el problema de reconocimiento de dígitos manuscritos del dataset MNIST antes de incorporar estrategias más avanzadas como Batch Normalization, Dropout o Data Augmentation.

### Arquitectura

* Dos bloques convolucionales (`Conv2D + MaxPooling`)
* Capas densas para clasificación
* Regularización mediante Dropout
* Función de activación ReLU
* Capa de salida Softmax para clasificación multiclase

### Objetivo de evaluación

Este modelo actuará como referencia para medir el impacto de las mejoras introducidas en las siguientes versiones y analizar la relación entre complejidad del modelo y rendimiento obtenido.


In [ ]:
# ============================================
# MODELO CNN BASE
# ============================================

model = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),

    tf.keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.25),

    tf.keras.layers.Dense(10, activation="softmax")
])

model.summary()

In [ ]:
# ============================================
# COMPILACIÓN
# ============================================

optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001
)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 🚀 Entrenamiento del modelo

El entrenamiento se realiza utilizando:

- Optimizador Adam
- Función de pérdida Sparse Categorical Crossentropy
- Batch size de 32
- Validación sobre conjunto hold-out

In [ ]:
# ============================================
# ENTRENAMIENTO
# ============================================

history = model.fit(
    X_train,
    Y_train,
    validation_data=(X_val, Y_val),
    epochs=20,
    batch_size=32
)

In [ ]:
model1_val_acc_max = max(history.history['val_accuracy'])
model1_params = model.count_params()

## 📈 Evaluación del rendimiento

Se analizan:

- Evolución de la pérdida
- Evolución de accuracy
- Capacidad de generalización
- Posible aparición de overfitting

In [ ]:
# ============================================
# CURVAS DE ENTRENAMIENTO
# ============================================

plt.figure(figsize=(10,5))

# Loss
plt.subplot(1,2,1)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.title("Loss")
plt.legend()

# Accuracy
plt.subplot(1,2,2)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.title("Accuracy")
plt.legend()

plt.show()

## 🔍 Análisis de errores

La matriz de confusión permite identificar:

- Clases con mayor dificultad
- Patrones de error
- Rendimiento por categoría

In [ ]:
# ============================================
# PREDICCIONES VALIDACIÓN
# ============================================

predictions = model.predict(X_val)

y_pred = np.argmax(predictions, axis=1)

cm = confusion_matrix(Y_val, y_pred)

plt.figure(figsize=(10,8))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
plt.show()

## &#x1F4CB; Predicciones

Se realizan las predicciones sobre el conjunto de test.

In [ ]:
# predicción del primer elemento del conjunto de test

model.predict(x_test[:1])

Como se observa, el método predict devuelve un array con el porcentaje de posibilidades de cada una de las 10 etiquetas (números del 0 al 9). Sería cuestión de quedarse con el elemento mayor, que representaría el mayor porcentaje.

Para la primera imagen del conjunto de test, el modelo predice un 2 como etiqueta de la imagen, con un porcentaje del 100 por 100 prácticamente.

A continuación se realiza, con ese mismo modelo, la predicción para las 28000 imágenes del conjunto de test:

In [ ]:
test_pred = [p.argmax() for p in model.predict(x_test[:])]

In [ ]:
# Se carga el dataset que habrá que modificar para subirlo a la competición de Kaggle.

df_sample_submission = pd.read_csv('data/sample_submission.csv')
print(f'Dimensión de df_sample_submission: {df_sample_submission.shape}')
df_sample_submission.head()

In [ ]:
df_sample_submission['Label'] = test_pred
df_sample_submission.head()

Este es el dataset que se sube a las competiciones de Kaggle para que el modelo sea evaluado y aparecer en la clasificación.

Se guarda el dataset para descargarlo en local y tener la opción de subirlo a Kaggle.

In [ ]:
df_sample_submission.to_csv("submission_cnn_base.csv", index=False)

# 🏗️ Modelo 2 · CNN Mejorada con Regularización y Data Augmentation

## Objetivo

Tras establecer una línea base mediante una CNN convencional, se implementa una versión mejorada orientada a aumentar la capacidad de generalización del modelo y reducir el riesgo de sobreajuste.

Para ello se incorporan técnicas ampliamente utilizadas en entornos de producción de Deep Learning, como Batch Normalization, Dropout y Data Augmentation. Estas estrategias permiten estabilizar el entrenamiento, mejorar la robustez frente a variaciones en los datos de entrada y favorecer una mejor capacidad de generalización sobre datos no vistos.

El propósito de este experimento es evaluar si la incorporación de técnicas de regularización y aumento de datos produce mejoras medibles respecto al modelo baseline.

### Mejoras introducidas

#### Batch Normalization

La normalización por lotes se incorpora después de las capas convolucionales para estabilizar la distribución de activaciones internas, acelerar la convergencia y facilitar el entrenamiento de la red.

#### Dropout

Se aplica Dropout en las capas densas para reducir la dependencia entre neuronas y minimizar el riesgo de sobreajuste.

#### Data Augmentation

Se generan variaciones ligeras de las imágenes de entrenamiento mediante:

* Rotaciones aleatorias suaves
* Zoom aleatorio moderado

Estas transformaciones permiten aumentar artificialmente la diversidad del conjunto de entrenamiento manteniendo la semántica de los dígitos.

### Objetivo de evaluación

Comparar el rendimiento obtenido frente al modelo baseline y analizar el impacto de las técnicas de regularización y aumento de datos sobre la capacidad de generalización del modelo.


In [ ]:
# ============================================
# DATA AUGMENTATION
# ============================================

data_augmentation = tf.keras.Sequential([

    # Rotaciones leves
    tf.keras.layers.RandomRotation(0.05),

    # Zoom ligero
    tf.keras.layers.RandomZoom(0.05)

])

# ============================================
# MODELO CNN
# ============================================

model2 = tf.keras.models.Sequential([

    tf.keras.layers.Input(shape=(28, 28, 1)),

    # Data augmentation
    data_augmentation,

    # ============================================
    # BLOQUE CONVOLUCIONAL 1
    # ============================================

    tf.keras.layers.Conv2D(32, (3,3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPooling2D(),

    # ============================================
    # BLOQUE CONVOLUCIONAL 2
    # ============================================

    tf.keras.layers.Conv2D(64, (3,3), padding="same"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.MaxPooling2D(),

    # ============================================
    # CLASIFICADOR
    # ============================================

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.20),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.20),

    tf.keras.layers.Dense(10, activation="softmax")
])

model2.summary()

In [ ]:
# ============================================
# COMPILACIÓN
# ============================================

optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001
)

model2.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# ============================================
# ENTRENAMIENTO
# ============================================

history = model2.fit(
    X_train,
    Y_train,
    validation_data=(X_val, Y_val),
    epochs=20,
    batch_size=32
)

In [ ]:
model2_val_acc_max = max(history.history['val_accuracy'])
model2_params = model2.count_params()

## 📈 Evaluación del rendimiento

Se analizan:

- Evolución de la pérdida
- Evolución de accuracy
- Capacidad de generalización
- Posible aparición de overfitting

In [ ]:
# ============================================
# CURVAS DE ENTRENAMIENTO
# ============================================

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)

plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")

plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)

plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")

plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()

plt.savefig(
    "images/training_curves.png",
    bbox_inches="tight"
)

plt.show()

## 🔍 Análisis de errores

La matriz de confusión permite identificar:

- Clases con mayor dificultad
- Patrones de error
- Rendimiento por categoría

In [ ]:
# ============================================
# PREDICCIONES VALIDACIÓN
# ============================================

predictions = model2.predict(X_val)

y_pred = np.argmax(predictions, axis=1)

cm = confusion_matrix(Y_val, y_pred)

plt.figure(figsize=(10,8))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
plt.show()

## &#x1F4CB; Predicciones

Se realizan las predicciones sobre el conjunto de test.

In [ ]:
# predicción del primer elemento del conjunto de test

model2.predict(x_test[:1])

In [ ]:
test_pred = [p.argmax() for p in model2.predict(x_test[:])]

In [ ]:
# Se carga el dataset que habrá que modificar para subirlo a la competición de Kaggle.

df_sample_submission = pd.read_csv('data/sample_submission.csv')
print(f'Dimensión de df_sample_submission: {df_sample_submission.shape}')
df_sample_submission.head()

In [ ]:
df_sample_submission['Label'] = test_pred
df_sample_submission.head()

In [ ]:
df_sample_submission.to_csv("submission_cnn_model_mejorada.csv", index=False)

## 🧠 Transfer Learning con MobileNetV2

Se evaluó un enfoque de transfer learning utilizando MobileNetV2
preentrenado sobre ImageNet.

Aunque esta estrategia suele ofrecer excelentes resultados en
problemas de clasificación de imágenes complejas, en este caso
el rendimiento fue inferior al obtenido mediante CNNs diseñadas
específicamente para MNIST.

La principal causa es la diferencia de dominio entre ImageNet
(imágenes RGB naturales) y MNIST (dígitos manuscritos
monocromáticos de 28x28 píxeles).

Este experimento confirma que los modelos preentrenados no siempre
son la mejor opción y que la selección de arquitectura debe
adaptarse al problema concreto.

In [ ]:
# ============================================
# ADAPTACIÓN DE DATOS PARA TRANSFER LEARNING
# ============================================

# X_train, X_test deben estar normalizados (0-1)
print("Shape original X_train:", X_train.shape)

# --------------------------------------------
# 1. Convertir a 3 canales (grayscale → RGB)
# --------------------------------------------
x_train_rgb = np.repeat(X_train, 3, axis=-1).astype("float32")
x_val_rgb = np.repeat(X_val, 3, axis=-1).astype("float32")
x_test_rgb = np.repeat(x_test, 3, axis=-1).astype("float32")

print("Después de convertir a RGB:", x_train_rgb.shape)

# --------------------------------------------
# 2. Redimensionar de forma segura (sin .numpy())
# --------------------------------------------
IMG_SIZE = 96

x_train_resized = tf.image.resize(
    x_train_rgb,
    (IMG_SIZE, IMG_SIZE)
)

x_val_resized = tf.image.resize(
    x_val_rgb,
    (IMG_SIZE, IMG_SIZE)
)

x_test_resized = tf.image.resize(
    x_test_rgb,
    (IMG_SIZE, IMG_SIZE)
)

# --------------------------------------------
# 3. Convertir a float32 (IMPORTANTE para RAM)
# --------------------------------------------
x_train_resized = tf.cast(x_train_resized, tf.float32)
x_val_resized = tf.cast(x_val_resized, tf.float32)
x_test_resized = tf.cast(x_test_resized, tf.float32)

print("Shape final train:", x_train_resized.shape)
print("Shape final val:", x_val_resized.shape)
print("Shape final test:", x_test_resized.shape)

# 🏗️ Modelo 3 - Uso de transfer learning con MobileNetV2

Se utiliza MobileNetV2 preentrenado en ImageNet como extractor de características.

- Se congela la base convolucional
- Se añade un clasificador personalizado
- Se aplica Dropout para regularización

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# ============================================
# MODELO BASE (TRANSFER LEARNING)
# ============================================

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False  # congelamos pesos

# ============================================
# HEAD DEL MODELO
# ============================================

model_tl = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax")
])

model_tl.summary()

## ⚙️ Compilación y entranamiento del modelo

Se utiliza el optimizador Adam y la función de pérdida estándar para clasificación multiclase.

In [ ]:
# ============================================
# COMPILACIÓN
# ============================================

model_tl.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================
# ENTRENAMIENTO
# ============================================

history = model_tl.fit(
    x_train_resized,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32
)

In [ ]:
model_tl_val_acc_max = max(history.history['val_accuracy'])
model_tl_params = model_tl.count_params()

## 📊 Evaluación del modelo

Se comparan las curvas de entrenamiento y validación para analizar:

- Convergencia del modelo
- Posible overfitting
- Rendimiento frente a CNN desde cero

In [ ]:
plt.plot(history.history["accuracy"], label="train_acc")
plt.plot(history.history["val_accuracy"], label="val_acc")
plt.title("MobileNetV2 - Accuracy")
plt.legend()
plt.show()

plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title("MobileNetV2 - Loss")
plt.legend()
plt.show()

## 🔍 Análisis de errores

La matriz de confusión permite identificar:

- Clases con mayor dificultad
- Patrones de error
- Rendimiento por categoría

In [ ]:
# ============================================
# PREDICCIONES VALIDACIÓN
# ============================================

predictions = model_tl.predict(x_val_resized)

y_pred = np.argmax(predictions, axis=1)

cm = confusion_matrix(Y_val, y_pred)

plt.figure(figsize=(10,8))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()
plt.show()

## &#x1F4CB; Predicciones

Se realizan las predicciones sobre el conjunto de test.

In [ ]:
test_pred = np.argmax(
    model_tl.predict(x_test_resized),
    axis=1
)

In [ ]:
df_sample_submission['Label'] = test_pred
df_sample_submission.head()

In [ ]:
df_sample_submission['Label'] = test_pred
df_sample_submission.head()

In [ ]:
df_sample_submission.to_csv("submission_tl.csv", index=False)

## 🧾 Conclusión del Transfer Learning
El modelo basado en MobileNetV2 permite:

Obtener una buena generalización incluso en un dataset simple
Reducir tiempo de entrenamiento gracias a pesos preentrenados
Comparar el rendimiento frente a CNN diseñadas desde cero
En problemas como MNIST, su uso no es estrictamente necesario, pero es útil como demostración de ingeniería de modelos avanzados.

# &#x1F4C8; Comparativa de Modelos

A continuación se muestran los mejores resultados de validación obtenidos por cada una de las arquitecturas evaluadas durante el proyecto.

La métrica utilizada para la comparación es el mejor **Accuracy de Validación** alcanzado durante el entrenamiento de cada modelo.

Esta comparativa permite analizar el impacto de las distintas estrategias de mejora incorporadas sobre el modelo baseline.

In [ ]:
# ============================================
# COMPARATIVA FINAL DE MODELOS
# ============================================

results = pd.DataFrame({
    "Modelo": [
        "CNN Base",
        "CNN Mejorada",
        "MobileNetV2"
    ],
    "Val Accuracy (%)": [
        round(model1_val_acc_max * 100, 2),
        round(model2_val_acc_max * 100, 2),
        round(model_tl_val_acc_max * 100, 2)
    ],
    "Parámetros": [
        f"{model1_params:,}",
        f"{model2_params:,}",
        f"{model_tl_params:,}"
    ]
})

results = results.sort_values(
    by="Val Accuracy (%)",
    ascending=False
)

fig, ax = plt.subplots(figsize=(8, 2))

ax.axis("off")

table = ax.table(
    cellText=results.values,
    colLabels=results.columns,
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

plt.savefig(
    "images/model_comparison.png",
    bbox_inches="tight",
    dpi=300
)

plt.show()

results

## Conclusiones

La CNN mejorada obtuvo el mejor rendimiento de validación, superando al modelo baseline gracias a la incorporación de técnicas de regularización y estabilización del entrenamiento.

Por el contrario, el enfoque basado en Transfer Learning mediante MobileNetV2 mostró un rendimiento inferior. Este resultado era esperable debido a la diferencia de dominio entre ImageNet (imágenes RGB naturales) y MNIST (dígitos manuscritos monocromáticos de baja resolución).

Los resultados confirman que, para problemas simples y altamente estructurados como MNIST, una CNN diseñada específicamente para el dominio puede superar a arquitecturas preentrenadas considerablemente más complejas.